# The ”Contract” & Few-Shot Learning

 #### The classifier mistakes ”News Reports” for ”SOS Calls”. 

In [ ]:

import sys
sys.path.append('..')

import pandas as pd
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model, should_use_reasoning_model
from IPython.display import Markdown, display

from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv()

data_path = "../data/data/Sample Messages.txt"
with open(data_path, "r") as f:
    messages_list = [line.strip() for line in f if line.strip()]
    
#print("Total messages:", len(messages_list))
#messages_list[:5]


In [42]:
examples = """

Example 1:
Message: We are trapped on the roof with 3 children and water is rising fast.
District: None
Intent: Rescue
Priority: High
Explanation: This is a life-threatening emergency with people trapped and immediate danger.

Example 2:
Message: Need food packets and drinking water at school shelter in Gampaha.
District: Gampaha
Intent: Supply
Priority: High
Explanation: The message requests essential supplies but does not indicate immediate danger to life.

Example 3:
Message: Breaking News: Kelani River level at 9m, flood warning issued.
District: Colombo
Intent: Info
Priority: Low
Explanation: This is a news report providing informational updates, not a direct request for help.

Example 4:
Message: Electricity restored in Kalutara town after floods.
District: Kalutara
Intent: Other
Priority: Low
Explanation: This is a general status update and does not require emergency response.

"""
model = pick_model('groq', 'general')
client = LLMClient('groq', model)

responses = []


for msg in messages_list:
    prompt_text, specs = render(
        "few_shot.v1",
        role="crisis message classifier",
        examples=examples,
        query=f"Message: {msg}",
        constraints=(
            "Classify the message strictly using the learned patterns."
            "Do not hallucinate any information."
            "Search the district and add it if it in the messege."
            "Because most of messages with district."
            "If district is not mentioned, respond with 'None'."
        ),
        
        format=(
            "District: [Name] | "
            "Intent: [Rescue/Supply/Info/Other] | "
            "Priority: [High/Low]"
        )
    )



    messages = [{"role": "user", "content": prompt_text}]

    response = client.chat(
        messages,
        temperature=0.0,
    )

    classification = response['text'].strip()
    responses.append({
        "Message": msg,
        "Classification": classification
    })

    log_llm_call('groq', model, 'few_shot', response['latency_ms'], response['usage'])


In [29]:
df = pd.DataFrame(responses)
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "classified_messages.xlsx"
df.to_excel(output_path, index=False)

output_path

WindowsPath('../outputs/classified_messages.xlsx')

In [38]:
test_cases = [
    "In Anuradhapura,We are in rooftop but we don't need help beacause water is decreasing now but we need someone if there water increasing here"
]

for test in test_cases:
    prompt_text, specs = render(
        "few_shot.v1",
        role="crisis message classifier",
        examples=examples,
        query=f"Message: {test}",
        format=(
            "District: [Name] | "
            "Intent: [Rescue/Supply/Info/Other] | "
            "Priority: [High/Low]"
        )
    )
    
    responses = client.chat(
        [{"role": "user", "content": prompt_text}],
        temperature=0.0,
    )
    
    print("Message:", test)
    print(responses['text'])
    print("-" * 60)

Message: In Anuradhapura,We are in rooftop but we don't need help beacause water is decreasing now but we need someone if there water increasing here
To classify the query message, I will analyze it based on the provided examples and constraints.

Message: In Anuradhapura, We are in rooftop but we don't need help because water is decreasing now but we need someone if there water increasing here

District: Anuradhapura (mentioned in the message)

Intent: The message initially states that they don't need help because the water is decreasing, but they want someone to be present in case the water increases. This implies that they are still in a vulnerable situation and want to be prepared for a potential emergency. Therefore, the intent is Rescue.

Priority: Although the water is decreasing, the message still expresses a need for someone to be present in case of an increase in water level. This indicates a potential for a life-threatening emergency, so the priority is High.

Output format: